Data Collection

restaurants & cafes in Auckland
tourist attractions in Auckland
parks in Auckland
shopping malls in Auckland

In [31]:
pip install requests pandas python-dotenv

Config (env, knobs, folders)

In [33]:
# === CONFIG ===
import os, time, json, math, requests, hashlib
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Load .env 
load_dotenv()
API_KEY = os.getenv("GOOGLE_API_KEY")  # <— standard name
assert API_KEY, "Set GOOGLE_API_KEY in your .env"

In [34]:
# ==== BUILD SEEDS FROM CHECKPOINTS 
import pandas as pd, os, glob, hashlib, re
from pathlib import Path

CHK = Path("data/checkpoints")
FINAL = Path("data/final")
FINAL.mkdir(parents=True, exist_ok=True)

# ---------- 1) PLACES SEED ----------

detail_files = sorted(CHK.glob("details_grid_classic_*.csv"))

places_frames = []
for f in detail_files:
    df = pd.read_csv(f)
    # Try to infer category from filename if not present as a column
    cat = re.sub(r"^details_grid_classic_|\.csv$", "", f.name)
    if "category" not in df.columns:
        df["category"] = cat.replace("_", " ")
    df["source_query"] = df.get("source_query", pd.Series([f"grid_classic_{cat}"] * len(df)))
    places_frames.append(df)

places_all = pd.concat(places_frames, ignore_index=True) if places_frames else pd.DataFrame()

keep_places = [
    "place_id","name","lat","lng","address","types","rating","user_ratings_total",
    "phone","website","opening_hours_present","source_query","category"
]
for c in keep_places:
    if c not in places_all.columns:
        places_all[c] = None

# Clean
places_seed = (places_all[keep_places]
               .dropna(subset=["place_id"])
               .drop_duplicates(subset=["place_id"])
               .reset_index(drop=True))

places_seed.to_csv(FINAL/"places_seed.csv", index=False)
print(" places_seed:", places_seed.shape, "-> data/final/places_seed.csv")

# ---------- 2) REVIEWS SEED ----------
review_files = sorted(CHK.glob("reviews_grid_classic_*.csv"))

rev_frames = []
for f in review_files:
    df = pd.read_csv(f)

  
    def ensure_col(df, out_col, candidates, default=None):
        for c in candidates:
            if c in df.columns:
                df[out_col] = df[c]
                return
        df[out_col] = default

    ensure_col(df, "place_id", ["place_id","placeId"])
    ensure_col(df, "text", ["text","review_text","content","comment","body"], "")
    ensure_col(df, "rating", ["rating","stars","score"])
    ensure_col(df, "lang", ["lang","language"], "en")
    ensure_col(df, "publish_time_utc", ["publish_time_utc","time_utc","time","timestamp"])  # your files had "time"
    ensure_col(df, "author_name", ["author_name","author","user"], None)
    ensure_col(df, "review_photo_url", ["review_photo_url","photo_url"], None)
    ensure_col(df, "review_id", ["review_id"], None)  # many rows already have this per screenshot

    mask = df.get("review_id") is None or df["review_id"].isna() | (df["review_id"].astype(str).str.strip()=="")
    if isinstance(mask, pd.Series) and mask.any():
        def mk_id(r):
            base = f"{r.get('place_id','')}|{r.get('publish_time_utc','')}|{(r.get('text','') or '')[:64]}"
            return hashlib.sha256(base.encode("utf-8")).hexdigest()[:24]
        df.loc[mask, "review_id"] = df[mask].apply(mk_id, axis=1)

    rev_frames.append(df[["review_id","place_id","text","rating","lang","publish_time_utc","author_name","review_photo_url"]])

reviews_all = pd.concat(rev_frames, ignore_index=True) if rev_frames else pd.DataFrame()

# Clean 
if not reviews_all.empty:
    reviews_all["text"] = (reviews_all["text"].fillna("").astype(str)
                           .str.replace(r"\s+"," ", regex=True).str.strip())
    reviews_all = reviews_all[reviews_all["text"]!=""]

    if "review_id" in reviews_all.columns:
        reviews_seed = reviews_all.drop_duplicates(subset=["review_id"]).reset_index(drop=True)
    else:
        reviews_seed = reviews_all.drop_duplicates(subset=["place_id","text","publish_time_utc"]).reset_index(drop=True)

    if "review_time" not in reviews_seed.columns and "publish_time_utc" in reviews_seed.columns:
        reviews_seed["review_time"] = reviews_seed["publish_time_utc"]
else:
    reviews_seed = pd.DataFrame(columns=["review_id","place_id","text","rating","lang","publish_time_utc","author_name","review_photo_url","review_time"])

reviews_seed.to_csv(FINAL/"reviews_seed.csv", index=False)
print(" reviews_seed:", reviews_seed.shape, "-> data/final/reviews_seed.csv")


✅ places_seed: (6252, 13) -> data/final/places_seed.csv
✅ reviews_seed: (27104, 9) -> data/final/reviews_seed.csv


In [35]:
# grid settings 
CITY = "Auckland"

# Crawl categories 
PLACE_TYPES = ["restaurant", "tourist attraction", "park", "shopping mall"]

# Rate limits 
NEARBY_NEXT_TOKEN_SLEEP = 2.0
DETAILS_SLEEP = 0.15

# Run flags 
RUN_TEXTSEARCH = False   
RUN_DETAILS    = False   

# Data folders 
PROC  = Path("data/processed"); PROC.mkdir(parents=True, exist_ok=True)
FINAL = Path("data/final");     FINAL.mkdir(parents=True, exist_ok=True)


Resilient HTTP Session (shared)

In [37]:
# One resilient session for all Google calls
SESSION = requests.Session()
retries = Retry(
    total=7, connect=7, read=7,
    backoff_factor=1.3,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods={"GET","POST"},
    raise_on_status=False,
)
adapter = HTTPAdapter(max_retries=retries, pool_connections=30, pool_maxsize=30)
SESSION.mount("https://", adapter)
SESSION.mount("http://", adapter)


Category mapper (final 4 buckets)

In [39]:
def map_to_category(google_types_str: str | None) -> str | None:
    """Collapse Google's many types to: restaurant, tourist attraction, park, shopping mall."""
    if google_types_str is None or (isinstance(google_types_str, float) and str(google_types_str) == "nan"):
        return None
    t = {x.strip().lower() for x in str(google_types_str).split(",")}
    if "shopping_mall" in t or "shopping mall" in t: return "shopping mall"
    if "tourist_attraction" in t or "tourist attraction" in t: return "tourist attraction"
    if "park" in t: return "park"
    if {"restaurant","cafe","coffee_shop","café","fast_food","meal_takeaway","meal_delivery"} & t: return "restaurant"
    return None


Text Search helpers(deduped, clean)

In [41]:
BASE_TEXT_SEARCH = "https://maps.googleapis.com/maps/api/place/textsearch/json"

def text_search(query, pagetoken=None):
    params = {"query": query, "key": API_KEY}
    if pagetoken: params["pagetoken"] = pagetoken
    r = SESSION.get(BASE_TEXT_SEARCH, params=params, timeout=(10, 30))
    r.raise_for_status()
    return r.json()

def collect_places_for_query(query, max_pages=3, sleep_between=NEARBY_NEXT_TOKEN_SLEEP):
    """Up to 3 pages. Returns minimal fields; 'google_types' not 'types'."""
    places = []
    token = None
    for _ in range(max_pages):
        if token:
            time.sleep(sleep_between)  # required delay for next_page_token
        data = text_search(query, pagetoken=token)
        results = data.get("results", [])
        for p in results:
            loc = (p.get("geometry", {}) or {}).get("location", {}) or {}
            places.append({
                "place_id": p.get("place_id"),
                "name": p.get("name"),
                "lat": loc.get("lat"),
                "lng": loc.get("lng"),
                "address": p.get("formatted_address"),
                "rating": p.get("rating"),
                "user_ratings_total": p.get("user_ratings_total"),
                "business_status": p.get("business_status"),
                "google_types": ",".join(p.get("types", [])),
                "source_category": query.split(" in ")[0].strip().lower(),  # store category only
            })
        token = data.get("next_page_token")
        if not token:
            break
    return places


Run Text Search (guarded)

In [43]:
if RUN_TEXTSEARCH:
    all_places = []
    for t in PLACE_TYPES:
        q = f"{t} in {CITY}"
        print("Fetching:", q)
        chunk = collect_places_for_query(q, max_pages=3)
        all_places.extend(chunk)
        print(f"  +{len(chunk)} (total={len(all_places)})")

    df_places = (pd.DataFrame(all_places)
                 .dropna(subset=["place_id"])
                 .drop_duplicates(subset=["place_id"])
                 .reset_index(drop=True))
    print("Unique places:", len(df_places))
    df_places.to_csv(PROC/"auckland_places_basic.csv", index=False)
else:
    basic_path = PROC/"auckland_places_basic.csv"
    assert basic_path.exists(), "Run text search once to create auckland_places_basic.csv"
    df_places = pd.read_csv(basic_path)
    print("Loaded basic places from disk:", len(df_places))


Loaded basic places from disk: 8130


Place Details (robust) + enrichment

In [45]:
BASE_PLACE_DETAILS = "https://maps.googleapis.com/maps/api/place/details/json"
DETAIL_FIELDS = (
    "name,place_id,geometry/location,formatted_address,types,"
    "rating,user_ratings_total,international_phone_number,website,opening_hours,reviews"
)

def place_details(pid):
    params = {"place_id": pid, "fields": DETAIL_FIELDS, "key": API_KEY}
    return SESSION.get(BASE_PLACE_DETAILS, params=params, timeout=(10, 30)).json()

def _details_with_backoff(pid, sleep_between=DETAILS_SLEEP):
    backoff = 1.0
    for _ in range(8):
        try:
            d = place_details(pid)
        except (requests.exceptions.SSLError,
                requests.exceptions.ConnectionError,
                requests.exceptions.ReadTimeout,
                requests.exceptions.ChunkedEncodingError):
            time.sleep(backoff); backoff = min(backoff*1.7, 20.0); continue
        status = d.get("status", "OK")
        if status == "OK": return d
        if status in ("OVER_QUERY_LIMIT", "UNKNOWN_ERROR"):
            time.sleep(backoff); backoff = min(backoff*1.7, 20.0); continue
        return d  
    return {"status":"SKIPPED"}

def enrich_places_with_details(df, limit=None, sleep_between=DETAILS_SLEEP):
    rows = df.to_dict(orient="records")
    if limit: rows = rows[:limit]
    enriched, reviews = [], []
    for p in rows:
        pid = p["place_id"]
        d = _details_with_backoff(pid, sleep_between=sleep_between)
        if d.get("status") != "OK":
            continue
        res = d.get("result") or {}
        loc = (res.get("geometry", {}) or {}).get("location", {}) or {}
        enriched.append({
            "place_id": pid,
            "name": res.get("name") or p.get("name"),
            "lat": loc.get("lat") or p.get("lat"),
            "lng": loc.get("lng") or p.get("lng"),
            "address": res.get("formatted_address") or p.get("address"),
            "google_types": ",".join(res.get("types", [])) or p.get("google_types"),
            "rating": res.get("rating"),
            "user_ratings_total": res.get("user_ratings_total"),
            "phone": res.get("international_phone_number"),
            "website": res.get("website"),
            "has_opening_hours": bool(res.get("opening_hours")),
            "source_category": p.get("source_category"),
        })
        for rv in (res.get("reviews") or []):
            text = (rv.get("text") or "").strip()
            tsec = rv.get("time")
            rid = hashlib.sha256(f"{pid}|{tsec}|{text[:64]}".encode("utf-8")).hexdigest()[:24]
            reviews.append({
                "review_id": rid,
                "place_id": pid,
                "author_name": rv.get("author_name"),
                "rating": rv.get("rating"),
                "text": text,
                "time": tsec,                  # unix seconds 
                "relative_time": rv.get("relative_time_description"),
                "language": rv.get("language"),
                
            })
        time.sleep(sleep_between)
    return pd.DataFrame(enriched), pd.DataFrame(reviews)

Pilot and Full enrichment (guarded)

In [47]:
if RUN_DETAILS:
    df_det_all, df_rev_all = enrich_places_with_details(df_places, limit=None)
    df_det_all.to_csv(PROC/"auckland_places_details.csv", index=False)
    df_rev_all.to_csv(PROC/"auckland_reviews.csv", index=False)
    print("Saved processed details & reviews.")
else:
    # Load processed details & reviews safel
    det_p = PROC / "auckland_places_details.csv"
    rev_p = PROC / "auckland_reviews.csv"

    def safe_load_csv(path, name):
        """Try to load a CSV, but return empty DataFrame if missing/empty."""
        if path.exists() and os.path.getsize(path) > 0:
            try:
                df = pd.read_csv(path)
                print(f"Loaded {name}: {df.shape}")
                return df
            except Exception as e:
                print(f" Failed to read {name} ({path}): {e}")
                return pd.DataFrame()
        else:
            print(f" {name} not found or empty at {path}")
            return pd.DataFrame()

    df_det_all = safe_load_csv(det_p, "places_details")
    df_rev_all = safe_load_csv(rev_p, "reviews")

    print("Processed DataFrames summary:")
    print(" - places_details:", df_det_all.shape)
    print(" - reviews:", df_rev_all.shape)


Loaded places_details: (1528, 13)
Loaded reviews: (6306, 9)
Processed DataFrames summary:
 - places_details: (1528, 13)
 - reviews: (6306, 9)


Clean, map categories, basic QC (single source of truth)

In [49]:
# Places
places = (df_det_all
          .dropna(subset=["place_id"])
          .drop_duplicates(subset=["place_id"])
          .copy())

for col in ["lat","lng","rating"]:
    if col in places.columns:
        places[col] = pd.to_numeric(places[col], errors="coerce")
if "user_ratings_total" in places.columns:
    places["user_ratings_total"] = pd.to_numeric(places["user_ratings_total"], errors="coerce").fillna(0).astype(int)

places["category"] = places["types"].apply(map_to_category)

keep_places = ["place_id","name","lat","lng","address","category","types",
               "rating","user_ratings_total","phone","website","has_opening_hours","source_category"]
places = places[[c for c in keep_places if c in places.columns]]


In [ ]:
#read last run time

import json, os
from datetime import datetime, timezone

LAST_RUN = None
runs_path = "data/final/runs.json"
if os.path.exists(runs_path) and os.path.getsize(runs_path) > 0:
    try:
        with open(runs_path, "r", encoding="utf-8") as f:
            LAST_RUN = json.load(f).get("last_successful_run_at")
    except Exception:
        pass

print("LAST_RUN:", LAST_RUN)

In [56]:
# LOAD SEEDS 
import os

seed_places = pd.read_csv("data/final/places_seed.csv")   
seed_reviews = pd.read_csv("data/final/reviews_seed.csv") 

if 'new_reviews_df' not in globals():
    new_reviews_df = pd.DataFrame(columns=seed_reviews.columns)

prev_reviews = pd.DataFrame()
prev_path = "data/final/reviews.csv"
if os.path.exists(prev_path) and os.path.getsize(prev_path) > 0:
    try:
        prev_reviews = pd.read_csv(prev_path)
    except Exception:
        pass

parts = [seed_reviews]
if not prev_reviews.empty: parts.append(prev_reviews)
if not new_reviews_df.empty: parts.append(new_reviews_df)

combined = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

if "review_id" in combined.columns:
    combined = combined.drop_duplicates(subset=["review_id"]).reset_index(drop=True)
else:
    combined = combined.drop_duplicates(subset=["place_id","text","publish_time_utc"]).reset_index(drop=True)

if "review_time" not in combined.columns and "publish_time_utc" in combined.columns:
    combined["review_time"] = combined["publish_time_utc"]

existing_path = "data/final/reviews.csv"
if not combined.empty:
    combined.to_csv(existing_path, index=False)
    upserted = len(combined)
    print(f" Saved reviews.csv with {upserted} rows")
else:
    if os.path.exists(existing_path) and os.path.getsize(existing_path)>0:
        print(" No reviews to write — keeping existing data/final/reviews.csv")
        upserted = 0
    else:
        scaffold_cols = ["review_id","place_id","text","rating","lang","publish_time_utc","review_time"]
        pd.DataFrame(columns=scaffold_cols).to_csv(existing_path, index=False)
        upserted = 0
        print(" Wrote empty scaffold reviews.csv")

from datetime import datetime, timezone
import json
run_info = {
    "last_successful_run_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "items_upserted": int(upserted),
    "notes": "seed+dynamic merge complete"
}
with open("data/final/runs.json", "w", encoding="utf-8") as f:
    json.dump(run_info, f, indent=2)


✅ Saved reviews.csv with 28917 rows


Final freeze 

In [ ]:
#fetch_new_reviews

seed_places = pd.read_csv("data/final/places_seed.csv")
place_ids = seed_places["place_id"].dropna().unique().tolist()

def fetch_latest_reviews(place_id, since_iso=None):
    return []

rows = []
for pid in place_ids:
    rows.extend(fetch_latest_reviews(pid, since_iso=LAST_RUN))

new_reviews_df = pd.DataFrame(rows)

print("Fetched new rows:", len(new_reviews_df))

In [62]:
import os, json
from datetime import datetime, timezone

FINAL = Path("data/final")
FINAL.mkdir(parents=True, exist_ok=True)

_candidates = ["combined", "reviews", "new_reviews_df", "df_rev_all"]
_final_reviews = None

for name in _candidates:
    if name in globals():
        val = eval(name)
        if isinstance(val, pd.DataFrame) and not val.empty:
            _final_reviews = val.copy()
            print(f"Using reviews from '{name}' → shape {_final_reviews.shape}")
            break

if _final_reviews is None:
    parts = []
    for p in [FINAL/"reviews_seed.csv", FINAL/"reviews.csv"]:
        if p.exists() and os.path.getsize(p) > 0:
            try:
                parts.append(pd.read_csv(p))
                print(f"Loaded fallback: {p}")
            except Exception as e:
                print(f" Could not read {p}: {e}")
    if parts:
        _final_reviews = pd.concat(parts, ignore_index=True)
        print(f"Using fallback combined reviews → shape {_final_reviews.shape}")
    else:
        _final_reviews = pd.DataFrame(columns=["review_id","place_id","text","rating","lang","publish_time_utc","review_time"])
        print(" No reviews available anywhere; creating empty scaffold.")

if "review_time" not in _final_reviews.columns and "publish_time_utc" in _final_reviews.columns:
    _final_reviews["review_time"] = _final_reviews["publish_time_utc"]

if "review_id" in _final_reviews.columns:
    _final_reviews = _final_reviews.drop_duplicates(subset=["review_id"]).reset_index(drop=True)
else:
    de_dupe_cols = [c for c in ["place_id","text","publish_time_utc"] if c in _final_reviews.columns]
    if de_dupe_cols:
        _final_reviews = _final_reviews.drop_duplicates(subset=de_dupe_cols).reset_index(drop=True)

reviews_path = FINAL/"reviews.csv"
if not _final_reviews.empty:
    _final_reviews.to_csv(reviews_path, index=False)
    upserted = len(_final_reviews)
    print(f" Saved {reviews_path} with {upserted} rows")
else:
    if reviews_path.exists() and os.path.getsize(reviews_path) > 0:
        print(f" No new/valid reviews; kept existing {reviews_path}")
        upserted = 0
    else:
        _final_reviews.to_csv(reviews_path, index=False)
        upserted = 0
        print(f" Wrote empty scaffold {reviews_path}")

_places_written = False
if "places" in globals() and isinstance(places, pd.DataFrame) and not places.empty:
    places.to_csv(FINAL/"places.csv", index=False)
    _places_written = True
elif (FINAL/"places_seed.csv").exists():
    try:
        pd.read_csv(FINAL/"places_seed.csv").to_csv(FINAL/"places.csv", index=False)
        _places_written = True
        print("Used places_seed.csv to write places.csv")
    except Exception as e:
        print(f" Could not write places.csv from seed: {e}")

print(f"places.csv written: {_places_written}")

run_info = {
    "last_successful_run_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "items_upserted": int(upserted),
    "notes": "final save (robust)"
}
with open(FINAL/"runs.json", "w", encoding="utf-8") as f:
    json.dump(run_info, f, indent=2)
print("Updated runs.json")


Using reviews from 'combined' → shape (28917, 13)
✅ Saved data\final\reviews.csv with 28917 rows
places.csv written: True
Updated runs.json
